In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# TFRClusterPermutationGLM Smoke Test (MS036, One Electrode)

This notebook runs an end-to-end smoke test of `TFRClusterPermutationGLM` on one subject (`MS036`) and one automatically selected valid electrode.

Model:
- Event: `feedback_start`
- Predictor of interest: `zrpe`
- Covariate: `zpm`
- Cluster permutations: `200`

> Note: this permutation count is intentionally low for a fast smoke run and is not intended for final inference.


In [ ]:
from pathlib import Path
import json
from datetime import datetime
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from IPython.display import display
import mne

from membandit.io import Subject
from membandit._impl.multivariate import TFRClusterPermutationGLM, _build_subject_regressors, run_batch_tfr_cluster_glm_pipeline, _default_filtered_subjects
from membandit.plotting_style import set_plot_style

warnings.filterwarnings("ignore")
set_plot_style()


In [ ]:
BASE_DIR = "/Volumes/T7 Shield"
LOAD_DIR = f"{BASE_DIR}/work/qasims01/MemoryBanditData/EMU"

SUBJECT = "MS036"
EVENT = "feedback_start"
PREDICTOR = "zrpe"
COVARIATES = ["zpm"]

N_PERMUTATIONS = 200
ALTERNATIVE = "two-sided"
ALPHA_CLUSTER = 0.05
ALPHA_TEST = 0.05
RANDOM_STATE = 42

ROI_PRIORITY = ["OFC", "HPC", "AMY", "ACC", "dlPFC", "dmPFC"]

OUTPUT_DIR = Path('/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-TFRClusterPermutationGLM-smoketest')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
subject_obj = Subject(SUBJECT, BASE_DIR, LOAD_DIR, verbose=False)

reg_df, trial_var = _build_subject_regressors(
    subject_obj=subject_obj,
    event=EVENT,
    predictor=PREDICTOR,
    covariates=COVARIATES,
    regressor_source="learn",
)

reg_df = reg_df[["trial", PREDICTOR] + COVARIATES].copy().sort_values("trial").reset_index(drop=True)

print("Regressor diagnostics")
print("- trial column used:", trial_var)
print("- rows:", len(reg_df))
print("- NaN counts:")
print(reg_df.isna().sum())
print("- trial range:", int(reg_df["trial"].min()), "to", int(reg_df["trial"].max()))


In [ ]:
def select_first_valid_electrode(elec_df, tfr_ch_names, roi_priority):
    ch_set = set(str(ch) for ch in tfr_ch_names)
    elec_df = elec_df.copy()
    elec_df["label"] = elec_df["label"].astype(str)

    for roi in roi_priority:
        labels = elec_df.loc[elec_df["salman_region"] == roi, "label"].tolist()
        for lab in labels:
            if lab in ch_set:
                return lab, roi

    for _, row in elec_df.iterrows():
        lab = str(row["label"])
        roi = str(row.get("salman_region", "unknown"))
        if lab in ch_set:
            return lab, roi

    raise ValueError("No electrode label overlap found between elec_df and TFR channels.")


tfr_obj = subject_obj._ensure_tfr_loaded(EVENT)
tfr = tfr_obj[0] if isinstance(tfr_obj, list) else tfr_obj

if np.asarray(tfr.data).ndim != 4:
    raise ValueError(f"Expected 4D TFR data (epochs, channels, freqs, times), got {np.asarray(tfr.data).shape}")

selected_ch, selected_roi = select_first_valid_electrode(subject_obj.elec_df, tfr.ch_names, ROI_PRIORITY)
ch_names_str = [str(ch) for ch in tfr.ch_names]
ch_idx = ch_names_str.index(selected_ch)

tfr_data = np.asarray(tfr.data[:, ch_idx, :, :], dtype=np.float64)
times_ms = np.asarray(tfr.times, dtype=np.float64) * 1000.0
freqs = np.asarray(tfr.freqs, dtype=np.float64)

print("Selected channel:", selected_ch)
print("Selected region:", selected_roi)
print("Raw electrode TFR shape (epochs, freqs, times):", tfr_data.shape)
print("Time range (ms):", float(times_ms.min()), "to", float(times_ms.max()))
print("Frequency range (Hz):", float(freqs.min()), "to", float(freqs.max()))


In [ ]:
trial_ix = reg_df["trial"].to_numpy(dtype=int) - 1  # 1-based -> 0-based
in_bounds = (trial_ix >= 0) & (trial_ix < tfr_data.shape[0])

reg_df_valid = reg_df.loc[in_bounds, ["trial", PREDICTOR] + COVARIATES].reset_index(drop=True)
trial_ix_valid = trial_ix[in_bounds]
tfr_data_aligned = tfr_data[trial_ix_valid, :, :]

predictor_df = reg_df_valid[[PREDICTOR] + COVARIATES].copy()
finite_mask = np.isfinite(predictor_df.to_numpy(dtype=np.float64)).all(axis=1)
dropped_predictor_nan = int((~finite_mask).sum())

predictor_df = predictor_df.loc[finite_mask].reset_index(drop=True)
trials_aligned = reg_df_valid.loc[finite_mask, "trial"].to_numpy(dtype=int)
tfr_data_aligned = tfr_data_aligned[finite_mask, :, :]

print("Alignment diagnostics")
print("- initial regressor rows:", len(reg_df))
print("- retained in-bounds rows:", len(reg_df_valid))
print("- dropped rows with predictor/covariate NaN:", dropped_predictor_nan)
print("- final aligned rows:", len(predictor_df))
print("- final aligned TFR shape:", tfr_data_aligned.shape)
print("- aligned trial range:", int(trials_aligned.min()), "to", int(trials_aligned.max()))

if len(predictor_df) < 20:
    raise ValueError(f"Hard fail: aligned rows < 20 (got {len(predictor_df)}).")


In [ ]:
assert PREDICTOR in predictor_df.columns, f"Missing predictor column: {PREDICTOR}"
for cov in COVARIATES:
    assert cov in predictor_df.columns, f"Missing covariate column: {cov}"

for col in predictor_df.columns:
    assert pd.api.types.is_numeric_dtype(predictor_df[col]), f"Column must be numeric: {col}"

assert len(predictor_df) == tfr_data_aligned.shape[0], "Predictor/TFR epoch count mismatch"
assert np.isfinite(predictor_df.to_numpy(dtype=np.float64)).all(), "Predictor matrix contains NaN/inf"
assert np.isfinite(tfr_data_aligned).all(), "TFR data contains NaN/inf"

print("Ready to run GLM")
print(f"- subject: {SUBJECT}")
print(f"- event: {EVENT}")
print(f"- channel: {selected_ch} ({selected_roi})")
print(f"- epochs: {tfr_data_aligned.shape[0]} | freqs: {tfr_data_aligned.shape[1]} | times: {tfr_data_aligned.shape[2]}")


In [ ]:
glm = TFRClusterPermutationGLM(
    tfr_data=tfr_data_aligned,
    predictor_data=predictor_df,
    target_var=PREDICTOR,
    ch_name=selected_ch,
    alternative=ALTERNATIVE,
    alpha_cluster=ALPHA_CLUSTER,
    alpha_test=ALPHA_TEST,
    standardize_X=True,
    random_state=RANDOM_STATE,
)

res = glm.run(n_permutations=N_PERMUTATIONS, n_jobs=1, verbose=0)
print("GLM run complete.")


In [ ]:
clusters = res["clusters"]
sig_clusters = res["significant_clusters"]

print("Smoke summary")
print(f"- subject: {SUBJECT}")
print(f"- channel: {selected_ch}")
print(f"- region: {selected_roi}")
print(f"- df_resid: {res['df_resid']}")
print(f"- tcrit: {res['tcrit']:.4f}")
print(f"- suprathreshold clusters: {len(clusters)}")
print(f"- significant clusters (p < {ALPHA_TEST}): {len(sig_clusters)}")

top_rows = []
for c in clusters[:5]:
    top_rows.append({
        "sign": c.sign,
        "pvalue": float(c.pvalue),
        "abs_mass": float(c.abs_mass),
        "freq_idx": tuple(int(v) for v in c.freq_idx),
        "time_idx": tuple(int(v) for v in c.time_idx),
        "n_pixels": int(c.n_pixels),
    })

cluster_table = pd.DataFrame(top_rows, columns=["sign", "pvalue", "abs_mass", "freq_idx", "time_idx", "n_pixels"])
display(cluster_table)


In [ ]:
def symmetric_norm(arr):
    vmax = float(np.nanmax(np.abs(arr)))
    if not np.isfinite(vmax) or vmax <= 0:
        return None
    return TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)


def build_significant_mask(shape, significant_clusters):
    mask = np.zeros(shape, dtype=bool)
    for c in significant_clusters:
        f0, f1 = c.freq_idx
        t0, t1 = c.time_idx
        mask[f0:f1+1, t0:t1+1] = True
    return mask


beta_map = np.asarray(res["beta_map"], dtype=np.float64)
t_map = np.asarray(res["t_map"], dtype=np.float64)
sig_mask = build_significant_mask(t_map.shape, sig_clusters)

extent = [float(times_ms.min()), float(times_ms.max()), float(freqs.min()), float(freqs.max())]
dt = float(np.median(np.diff(times_ms))) if len(times_ms) > 1 else 1.0
df = float(np.median(np.diff(freqs))) if len(freqs) > 1 else 1.0

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)

# Panel A: beta map
im0 = axes[0].imshow(
    beta_map,
    origin="lower",
    aspect="auto",
    extent=extent,
    cmap="RdBu_r",
    norm=symmetric_norm(beta_map),
)
axes[0].set_title("A. Beta Map (zrpe)")
axes[0].set_xlabel("Time (ms)")
axes[0].set_ylabel("Frequency (Hz)")
cb0 = fig.colorbar(im0, ax=axes[0], shrink=0.9)
cb0.set_label("Beta")

# Panel B: t map
im1 = axes[1].imshow(
    t_map,
    origin="lower",
    aspect="auto",
    extent=extent,
    cmap="RdBu_r",
    norm=symmetric_norm(t_map),
)
axes[1].set_title("B. T Map (zrpe)")
axes[1].set_xlabel("Time (ms)")
axes[1].set_ylabel("Frequency (Hz)")
cb1 = fig.colorbar(im1, ax=axes[1], shrink=0.9)
cb1.set_label("t")

# Panel C: significant-cluster overlay
im2 = axes[2].imshow(
    t_map,
    origin="lower",
    aspect="auto",
    extent=extent,
    cmap="Greys",
    alpha=0.5,
)
overlay = np.ma.masked_where(~sig_mask, sig_mask.astype(float))
axes[2].imshow(
    overlay,
    origin="lower",
    aspect="auto",
    extent=extent,
    cmap="viridis",
    alpha=0.7,
)

for c in sig_clusters:
    f0, f1 = c.freq_idx
    t0, t1 = c.time_idx
    x0 = float(times_ms[t0])
    y0 = float(freqs[f0])
    width = float(times_ms[t1] - times_ms[t0] + dt)
    height = float(freqs[f1] - freqs[f0] + df)
    rect = Rectangle((x0, y0), width, height, fill=False, edgecolor="yellow", linewidth=1.5)
    axes[2].add_patch(rect)

if len(sig_clusters) == 0:
    axes[2].text(
        0.5, 0.5, "No significant clusters", transform=axes[2].transAxes,
        ha="center", va="center", fontsize=10, color="black",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="gray")
    )

axes[2].set_title("C. Significant Cluster Mask")
axes[2].set_xlabel("Time (ms)")
axes[2].set_ylabel("Frequency (Hz)")
cb2 = fig.colorbar(im2, ax=axes[2], shrink=0.9)
cb2.set_label("t (background)")

fig.suptitle(
    f"TFR GLM Smoke | subj={SUBJECT} | ch={selected_ch} ({selected_roi}) | event={EVENT} | perms={N_PERMUTATIONS}",
    fontsize=12,
)

safe_ch = str(selected_ch).replace("/", "-").replace(" ", "_")
fig_path = OUTPUT_DIR / f"tfr_glm_smoke_{SUBJECT}_{safe_ch}_{EVENT}.png"
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved smoke plot: {fig_path}")
plt.show()


In [ ]:
null_vals = np.asarray(res["null_max_abs_masses"], dtype=np.float64)
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.hist(null_vals, bins=30, color="steelblue", alpha=0.8, edgecolor="white")

if len(clusters) > 0:
    max_obs_mass = float(max(c.abs_mass for c in clusters))
    ax.axvline(max_obs_mass, color="crimson", linestyle="--", linewidth=2,
               label=f"max observed |cluster mass| = {max_obs_mass:.2f}")
    ax.legend(loc="best", frameon=False)
else:
    ax.text(0.5, 0.9, "No observed clusters", transform=ax.transAxes, ha="center", va="center")

ax.set_title("Null Distribution: Max |Cluster Mass|")
ax.set_xlabel("Max |cluster mass| per permutation")
ax.set_ylabel("Count")
sns.despine()
plt.tight_layout()
plt.show()


## Batch All-Subject / All-Electrode Run (Checkpointed, Memory-Safe)

- Processes subjects sequentially (memory-safe for large TFR files).
- Uses per-subject checkpoints for resumable execution.
- Saves plots only for electrodes with >=1 significant cluster by default.
- This run can take a long time at full scale.


In [ ]:
# Batch defaults (aligned with smoke settings)
BATCH_EVENT = "feedback_start"
BATCH_PREDICTOR = "zrpe"
BATCH_COVARIATES = ["zpm", "pgamble", "preward"]
BATCH_N_PERMUTATIONS = 200
BATCH_ALPHA_CLUSTER = 0.05
BATCH_ALPHA_TEST = 0.05
BATCH_RANDOM_STATE = 42
BATCH_MIN_ALIGNED_TRIALS = 20

# Execution controls
BATCH_RESUME = True
BATCH_OVERWRITE = False
BATCH_MAX_SUBJECTS = None  # set 1 or 2 for staged tests
BATCH_MAX_ELECTRODES_PER_SUBJECT = None
BATCH_SAVE_NON_SIG_PLOTS = False

# Timestamped run directory (change to fixed path to resume across sessions)
BATCH_RUN_DIR = OUTPUT_DIR / f"batch_tfr_glm_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print("Batch run directory:", BATCH_RUN_DIR)


In [ ]:
subjects = _default_filtered_subjects(BASE_DIR)

# Optional manual override examples:
# subjects = ["MS036"]
# subjects = subjects[:2]

print(f"Filtered subjects: {len(subjects)}")
print("First 10:", subjects[:10])


In [ ]:
checkpoints_dir = BATCH_RUN_DIR / "checkpoints"
existing_audit_rows = 0
if checkpoints_dir.exists():
    for pth in checkpoints_dir.glob("*_audit.csv"):
        try:
            existing_audit_rows += len(pd.read_csv(pth))
        except Exception:
            pass

print("Preflight")
print("- subjects requested:", len(subjects))
print("- output run dir:", BATCH_RUN_DIR)
print("- resume:", BATCH_RESUME, "| overwrite:", BATCH_OVERWRITE)
print("- max_subjects:", BATCH_MAX_SUBJECTS)
print("- max_electrodes_per_subject:", BATCH_MAX_ELECTRODES_PER_SUBJECT)
print("- save_non_sig_plots:", BATCH_SAVE_NON_SIG_PLOTS)
print("- existing audit rows in run dir:", existing_audit_rows)


In [ ]:
print("- output run dir:", BATCH_RUN_DIR)


In [ ]:
batch_out = run_batch_tfr_cluster_glm_pipeline(
    subjects=subjects,
    base_dir=BASE_DIR,
    load_dir=LOAD_DIR,
    run_dir=str(BATCH_RUN_DIR),
    event=BATCH_EVENT,
    predictor=BATCH_PREDICTOR,
    covariates=BATCH_COVARIATES,
    n_permutations=BATCH_N_PERMUTATIONS,
    alpha_cluster=BATCH_ALPHA_CLUSTER,
    alpha_test=BATCH_ALPHA_TEST,
    alternative="two-sided",
    min_aligned_trials=BATCH_MIN_ALIGNED_TRIALS,
    random_state=BATCH_RANDOM_STATE,
    overwrite=BATCH_OVERWRITE,
    resume=BATCH_RESUME,
    max_subjects=BATCH_MAX_SUBJECTS,
    max_electrodes_per_subject=BATCH_MAX_ELECTRODES_PER_SUBJECT,
    save_non_sig_plots=BATCH_SAVE_NON_SIG_PLOTS,
)

sig_df = batch_out["sig_electrodes"]
region_df = batch_out["region_counts"]
audit_df = batch_out["audit"]

print("Batch run complete.")
print("- sig rows:", len(sig_df))
print("- region rows:", len(region_df))
print("- audit rows:", len(audit_df))


In [ ]:
sig_csv = BATCH_RUN_DIR / "sig_electrodes.csv"
region_csv = BATCH_RUN_DIR / "region_counts.csv"
audit_csv = BATCH_RUN_DIR / "all_audit.csv"
meta_json = BATCH_RUN_DIR / "run_metadata.json"

print("Artifacts")
for p in [sig_csv, region_csv, audit_csv, meta_json]:
    print(f"- {p.name}: exists={p.exists()}")

if sig_csv.exists():
    print("sig_electrodes.csv rows:", len(pd.read_csv(sig_csv)))
if region_csv.exists():
    print("region_counts.csv rows:", len(pd.read_csv(region_csv)))
if audit_csv.exists():
    print("all_audit.csv rows:", len(pd.read_csv(audit_csv)))

if meta_json.exists():
    with open(meta_json, "r", encoding="utf-8") as f:
        meta = json.load(f)
    print(json.dumps(meta, indent=2))


In [ ]:
sig_view_cols = ["subject", "electrode", "region", "n_sig_clusters", "min_sig_p", "plot_path"]
region_view_cols = ["region", "n_sig_electrodes", "n_subjects_with_sig"]

print("Significant electrodes (core output)")
if len(sig_df) > 0:
    display(sig_df.loc[:, [c for c in sig_view_cols if c in sig_df.columns]])
else:
    print("No significant electrodes found.")

print("\nRegion counts (>=1 significant cluster per electrode)")
if len(region_df) > 0:
    display(region_df.loc[:, [c for c in region_view_cols if c in region_df.columns]])
else:
    print("No region counts available.")


In [ ]:
if len(region_df) > 0:
    region_plot_df = region_df.sort_values("n_sig_electrodes", ascending=False).copy()
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    sns.barplot(data=region_plot_df, x="region", y="n_sig_electrodes", color="steelblue", ax=ax)
    ax.set_title("Significant Electrodes by Region")
    ax.set_xlabel("Region")
    ax.set_ylabel("# Electrodes with >=1 Significant Cluster")
    ax.tick_params(axis="x", rotation=45)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print("Skipping region bar plot: region_df is empty.")

if len(sig_df) > 0 and "subject" in sig_df.columns:
    subj_counts = sig_df.groupby("subject")["electrode"].nunique().sort_values(ascending=False).reset_index(name="n_sig_electrodes")
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    sns.barplot(data=subj_counts, x="subject", y="n_sig_electrodes", color="darkorange", ax=ax)
    ax.set_title("Significant Electrodes by Subject")
    ax.set_xlabel("Subject")
    ax.set_ylabel("# Significant Electrodes")
    ax.tick_params(axis="x", rotation=45)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print("Skipping subject bar plot: sig_df is empty.")


### Resume Workflow Notes

- Continue interrupted run: keep the same `BATCH_RUN_DIR`, set `BATCH_RESUME=True`, `BATCH_OVERWRITE=False`.
- Force full rerun in same folder: set `BATCH_OVERWRITE=True` (or choose a new `BATCH_RUN_DIR`).
- Stage safely: start with `BATCH_MAX_SUBJECTS=1`, then `2`, then `None`.
- If you want true cross-session resume, set `BATCH_RUN_DIR` to a fixed non-timestamp path.
